In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import matplotlib.pyplot as plt

C:\Users\Admin\anaconda3\lib\site-packages\numpy\_distributor_init.py:30: UserWarning: loaded more than 1 DLL from .libs:
C:\Users\Admin\anaconda3\lib\site-packages\numpy\.libs\libopenblas.XWYDX2IKJW2NMTWSFYNGFUWKQU3LYTCZ.gfortran-win_amd64.dll
C:\Users\Admin\anaconda3\lib\site-packages\numpy\.libs\libopenblas64__v0.3.21-gcc_10_3_0.dll
  warnings.warn("loaded more than 1 DLL from .libs:"


In [2]:
df = pd.read_csv(
    r"C:\Users\Admin\Desktop\PROJECT FILE\cybersecurity_intrusion_data.csv"
)

print("Dataset Shape:", df.shape)

df.head()

Dataset Shape: (9537, 11)


,session_id,network_packet_size,protocol_type,login_attempts,session_duration,encryption_used,ip_reputation_score,failed_logins,browser_type,unusual_time_access,attack_detected
0,SID_00001,599,TCP,4,492.983263,DES,0.606818,1,Edge,0,1
1,SID_00002,472,TCP,3,1557.996461,DES,0.301569,0,Firefox,0,0
2,SID_00003,629,TCP,3,75.044262,DES,0.739164,2,Chrome,0,1
3,SID_00004,804,UDP,4,601.248835,DES,0.123267,0,Unknown,0,1
4,SID_00005,453,TCP,5,532.540888,AES,0.054874,1,Firefox,0,0


In [3]:
def data_quality_report(df):

    report = {
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Missing Values": df.isnull().sum().sum(),
        "Duplicate Rows": df.duplicated().sum(),
        "Data Types": df.dtypes.astype(str).to_dict()
    }

    return report


quality_report = data_quality_report(df)

quality_report

{'Rows': 9537,
 'Columns': 11,
 'Missing Values': 0,
 'Duplicate Rows': 0,
 'Data Types': {'session_id': 'object',
  'network_packet_size': 'int64',
  'protocol_type': 'object',
  'login_attempts': 'int64',
  'session_duration': 'float64',
  'encryption_used': 'object',
  'ip_reputation_score': 'float64',
  'failed_logins': 'int64',
  'browser_type': 'object',
  'unusual_time_access': 'int64',
  'attack_detected': 'int64'}}

In [4]:
def handle_missing_values(df):

    df = df.copy()

    for col in df.columns:

        if df[col].dtype in ["int64", "float64"]:
            df[col] = df[col].fillna(df[col].median())

        else:
            df[col] = df[col].fillna(df[col].mode()[0])

    return df


df_clean = handle_missing_values(df)

print("Missing Values After Cleaning:")
print(df_clean.isnull().sum().sum())

Missing Values After Cleaning:
0


In [5]:
duplicates_before = df_clean.duplicated().sum()

df_clean = df_clean.drop_duplicates()

duplicates_after = df_clean.duplicated().sum()

print("Duplicates Before:", duplicates_before)
print("Duplicates After:", duplicates_after)

Duplicates Before: 0
Duplicates After: 0


In [6]:
def detect_outliers(df):

    outlier_summary = {}

    numeric_cols = df.select_dtypes(include=np.number).columns

    for col in numeric_cols:

        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)

        IQR = Q3 - Q1

        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        count = ((df[col] < lower) | (df[col] > upper)).sum()

        outlier_summary[col] = int(count)

    return pd.DataFrame(
        outlier_summary.items(),
        columns=["Feature", "Outliers"]
    )


outlier_report = detect_outliers(df_clean)

outlier_report

,Feature,Outliers
0,network_packet_size,37
1,login_attempts,206
2,session_duration,418
3,ip_reputation_score,21
4,failed_logins,323
5,unusual_time_access,1430
6,attack_detected,0


In [7]:
numeric_cols = df_clean.select_dtypes(include=np.number).columns

normalizer = MinMaxScaler()

df_normalized = df_clean.copy()

df_normalized[numeric_cols] = normalizer.fit_transform(
    df_clean[numeric_cols]
)

df_normalized.head()

,session_id,network_packet_size,protocol_type,login_attempts,session_duration,encryption_used,ip_reputation_score,failed_logins,browser_type,unusual_time_access,attack_detected
0,SID_00001,0.438165,TCP,0.250000,0.068497,DES,0.655587,0.2,Edge,0.0,1.0
1,SID_00002,0.334152,TCP,0.166667,0.216623,DES,0.324443,0.0,Firefox,0.0,0.0
2,SID_00003,0.462735,TCP,0.166667,0.010368,DES,0.799160,0.4,Chrome,0.0,1.0
3,SID_00004,0.606061,UDP,0.250000,0.083555,DES,0.131015,0.0,Unknown,0.0,1.0
4,SID_00005,0.318591,TCP,0.333333,0.073998,AES,0.056820,0.2,Firefox,0.0,0.0


In [8]:
numeric_cols = df_clean.select_dtypes(include=np.number).columns

scaler = StandardScaler()

df_standardized = df_clean.copy()

df_standardized[numeric_cols] = scaler.fit_transform(
    df_clean[numeric_cols]
)

df_standardized.head()

,session_id,network_packet_size,protocol_type,login_attempts,session_duration,encryption_used,ip_reputation_score,failed_logins,browser_type,unusual_time_access,attack_detected
0,SID_00001,0.496899,TCP,-0.016346,-0.381125,DES,1.554930,-0.500779,Edge,-0.419989,1.112040
1,SID_00002,-0.143322,TCP,-0.525794,0.972960,DES,-0.168029,-1.467959,Firefox,-0.419989,-0.899248
2,SID_00003,0.648132,TCP,-0.525794,-0.912503,DES,2.301950,0.466400,Chrome,-0.419989,1.112040
3,SID_00004,1.530327,UDP,-0.016346,-0.243473,DES,-1.174443,-1.467959,Unknown,-0.419989,1.112040
4,SID_00005,-0.239103,TCP,0.493102,-0.330830,AES,-1.560484,-0.500779,Firefox,-0.419989,-0.899248


In [14]:
class PreprocessingService:

    def __init__(self, dataframe):
        self.df = dataframe.copy()

    def process(self):

        report = {}

        report["original_shape"] = self.df.shape

        self.df = handle_missing_values(self.df)

        duplicates_before = self.df.duplicated().sum()

        self.df = self.df.drop_duplicates()

        report["duplicates_removed"] = int(duplicates_before)

        report["final_shape"] = self.df.shape

        report["outliers"] = detect_outliers(self.df)

        return self.df, report
    service = PreprocessingService(df)

cleaned_df, report = service.process()

cleaned_df.head(20)

,session_id,network_packet_size,protocol_type,login_attempts,session_duration,encryption_used,ip_reputation_score,failed_logins,browser_type,unusual_time_access,attack_detected
0,SID_00001,599,TCP,4,492.983263,DES,0.606818,1,Edge,0,1
1,SID_00002,472,TCP,3,1557.996461,DES,0.301569,0,Firefox,0,0
2,SID_00003,629,TCP,3,75.044262,DES,0.739164,2,Chrome,0,1
3,SID_00004,804,UDP,4,601.248835,DES,0.123267,0,Unknown,0,1
4,SID_00005,453,TCP,5,532.540888,AES,0.054874,1,Firefox,0,0
5,SID_00006,453,UDP,5,380.471550,AES,0.422486,2,Chrome,1,0
6,SID_00007,815,ICMP,4,728.107165,AES,0.413772,1,Chrome,0,1
7,SID_00008,653,TCP,3,12.599906,DES,0.097719,3,Chrome,1,1
8,SID_00009,406,TCP,2,542.558895,None,0.294580,0,Chrome,1,0
9,SID_00010,608,UDP,6,531.944107,None,0.424117,1,Chrome,0,0
